# Tensile Test (TTO): from data entry to RDF

This notebook shows how to describe a uniaxial tensile test and its measured
results, and convert that description into a standardised, machine-readable
RDF graph using the
[Tensile Test Ontology (TTO)](https://w3id.org/pmd/tto/) and the
[Platform MaterialDigital Core Ontology (PMDCo)](https://w3id.org/pmd/co/).

**You only need to edit one file:** `docs/example.input.json`. Everything
else is automatic.


## TTO graph pattern

This schema is self-contained: it does not extend a base schema.
The graph pattern follows the
[TTO reference data](https://github.com/materialdigital/tensile-test-ontology).

Each measured value (result or condition) produces three linked nodes:

```
TensileTestingProcess  (pmdco:PMD_0000974)
  BFO_0000055  ─────────────────────────► TensileTestMethod (tto:TTO_0000054)
  OBI_0000293  ─────────────────────────► Specimen IRI
  PMD_0000009  ─────────────────────────► Quality node  [× 0..N]
    IAO_0000417  ─────────────────────────► _smd
  OBI_0000299  ─────────────────────────► _smd (IAO_0000109)  [× 0..N]
    IAO_0000136  ─────────────────────────► Quality node
    OBI_0001938  ─────────────────────────► _svs (OBI_0001931)
      PMD_0000006  ────────────────────────── xsd:float value
      IAO_0000039  ────────────────────────── QUDT unit IRI
      OBI_0001927, IAO_0000136  ───────────► Quality node
```


## What the notebook does

```
example.input.json
  │  specimen IRI, test conditions, measured properties
  │
  ▼
Transform
  │  converts plain JSON to a structured OO-LD document
  │
  ▼
RDF graph
  │  machine-readable, ontology-aligned triples
  │
  ▼
SHACL validation  →  confirms the graph is structurally correct
SPARQL inspect    →  shows the test results captured in the graph
```


## Environment setup

If you are viewing this notebook on GitHub and want to run it locally, clone
the repository first so that all schema files and example inputs are present:

```bash
git clone https://github.com/Semantic-Dataspace/semantic-schemas.git
cd semantic-schemas
```

Then create a virtual environment and start Jupyter:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install semantic-schemas jupyterlab
jupyter lab
```

Open this notebook from the `docs/` subfolder inside JupyterLab.

In [1]:
%pip install -q semantic-schemas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, pathlib, rdflib
from semantic_schemas import Schema

HERE       = pathlib.Path().resolve()          # docs/
SCHEMA     = HERE.parent                       # tensile-test/TTO/

schema = Schema(SCHEMA)

print("Schema :", "/".join(SCHEMA.parts[-3:]))


Schema : step/tensile-test/TTO


## Step 1: Describe your tensile test

Edit `docs/example.input.json` with your data, then run this cell to load it.

| Field | Required | Description |
|---|---|---|
| `test_name` | yes | A name for this test |
| `specimen_iri` | yes | IRI of the specimen tested (must exist in the knowledge graph) |
| `test_standard` | no | Standard followed, e.g. `"ISO 6892-1"` |
| `strain_rate` | no | Strain or crosshead displacement rate |
| `strain_rate_unit` | no | Unit for strain rate; defaults to `"1/s"` |
| `temperature` | no | Test temperature in °C |
| `results` | no | List of measured properties: `property`, `value`, `unit` |

Supported property names: `YieldStrength`, `TensileStrength`,
`PercentageElongationAfterFracture`, `PercentageReductionOfArea`,
`UpperYieldStrength`, `LowerYieldStrength`, `ProofStrength`,
`PercentagePermanentElongation`.

Supported units: `"MPa"`, `"GPa"` for stress properties; `"%"` for
elongation and reduction of area.

In [3]:
simplified = json.loads((HERE / "example.input.json").read_text())

print(json.dumps(simplified, indent=2))

{
  "test_name": "Tensile test 316L bar 1",
  "specimen_iri": "https://example.org/specimens/316L-tensile-bar-1",
  "test_standard": "ISO 6892-1",
  "strain_rate": 0.00025,
  "temperature": 23,
  "results": [
    {
      "property": "YieldStrength",
      "value": 310,
      "unit": "MPa"
    },
    {
      "property": "TensileStrength",
      "value": 620,
      "unit": "MPa"
    },
    {
      "property": "PercentageElongationAfterFracture",
      "value": 40,
      "unit": "%"
    },
    {
      "property": "PercentageReductionOfArea",
      "value": 65,
      "unit": "%"
    }
  ]
}


## Step 2: Convert to OO-LD

The transform converts your plain input into a structured OO-LD document
following the [TTO measurement pattern](https://github.com/materialdigital/tensile-test-ontology).

Each measured value (result or condition) produces three linked nodes:

- A **quality instance** typed to the TTO class (e.g. `tto:TTO_0000009` for yield
  strength), with an `iao:IAO_0000417` (`is quality measured as`) link to the datum.
- A **Scalar Measurement Datum** (`obo:IAO_0000109`) that is the specified output of
  the process, carrying an `is_about` link back to the quality instance.
- A **Scalar Value Specification** (`obo:OBI_0001931`) embedded in the datum via
  `obi:OBI_0001938`, carrying the numeric value and unit.

Condition qualities (strain rate, temperature) appear in `has_process_attribute`
and share the same datum chain.

In [4]:
oold_doc = schema.transform(simplified)

print(json.dumps(oold_doc, indent=2))

{
  "conforms_to": "https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/characterization/step/tensile-test/TTO/#v3.0.0",
  "type": "pmdco:PMD_0000974",
  "id": "tensile-test-tensile-test-316l-bar-1",
  "label": "Tensile test 316L bar 1",
  "realizes": [
    {
      "type": "tto:TTO_0000054",
      "id": "tensile-test-tensile-test-316l-bar-1-method",
      "label": "ISO 6892-1"
    }
  ],
  "has_specified_input": [
    "https://example.org/specimens/316L-tensile-bar-1"
  ],
  "has_process_attribute": [
    {
      "type": "tto:TTO_0000051",
      "id": "tensile-test-tensile-test-316l-bar-1-strain-rate",
      "is_quality_measured_as": "tensile-test-tensile-test-316l-bar-1-strain-rate-smd"
    },
    {
      "type": "pmdco:PMD_0000967",
      "id": "tensile-test-tensile-test-316l-bar-1-temperature",
      "is_quality_measured_as": "tensile-test-tensile-test-316l-bar-1-temperature-smd"
    }
  ],
  "measured_properties": [
    {
      "type": "obo:IAO_0000109",
      

## Step 3: Convert to RDF

The OO-LD document is parsed into an RDF graph using the ontology context
from `specs/schema.oold.yaml`. The context maps every field to its precise
ontology IRI, for example:

> **Tip:** Pass your own namespace as `base` to `to_graph()` so the generated
> IRIs use your data namespace instead of the schema's internal `@base`.
> The cell below uses `"https://example.org/"` as a placeholder.

| JSON field | Ontology IRI |
|---|---|
| `type` (process) | `rdf:type` → `pmdco:PMD_0000974` |
| `realizes` | `BFO_0000055` → TensileTestMethod (`tto:TTO_0000054`) |
| `has_specified_input` | `OBI_0000293` (has_specified_input) |
| `has_process_attribute` | `pmdco:PMD_0000009` → condition quality nodes |
| `is_quality_measured_as` | `IAO_0000417` → ScalarMeasurementDatum IRI |
| `measured_properties` | `OBI_0000299` → ScalarMeasurementDatum nodes (`IAO_0000109`) |
| `is_about` | `IAO_0000136` → quality instance IRI |
| `has_value_spec` | `OBI_0001938` → ScalarValueSpecification (`OBI_0001931`) |
| `result_value` | `pmdco:PMD_0000006` typed `xsd:float` |
| `result_unit` | `IAO_0000039` → QUDT unit IRI |
| `specifies_value_of` | `OBI_0001927` → quality instance IRI |

In [5]:
BASE_IRI = "https://example.org/"  # replace with your own namespace in production

flat = schema.to_graph(simplified, base=BASE_IRI)

print(f'Graph contains {len(flat)} triples.\n')
print(flat.serialize(format='turtle'))

Graph contains 74 triples.

@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix obo: <http://purl.obolibrary.org/obo/> .
@prefix pmdco: <https://w3id.org/pmd/co/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix tto: <https://w3id.org/pmd/tto/> .
@prefix unit: <http://qudt.org/vocab/unit/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://example.org/tensile-test-tensile-test-316l-bar-1> a pmdco:PMD_0000974 ;
    rdfs:label "Tensile test 316L bar 1" ;
    obo:BFO_0000055 <https://example.org/tensile-test-tensile-test-316l-bar-1-method> ;
    obo:OBI_0000293 <https://example.org/specimens/316L-tensile-bar-1> ;
    obo:OBI_0000299 <https://example.org/tensile-test-tensile-test-316l-bar-1-percentageelongationafterfracture-smd>,
        <https://example.org/tensile-test-tensile-test-316l-bar-1-percentagereductionofarea-smd>,
        <https://example.org/tensile-test-tensile-test-316l-bar-1-strain-rate-smd>,
        <https://example.org/tensile-test-tensile-

In [6]:
out_ttl = HERE / "output_tensile_test.ttl"
out_ttl.write_text(flat.serialize(format="turtle"))
print(f"Written to {out_ttl.name}")

Written to output_tensile_test.ttl


## Step 4: Validate against SHACL shapes

This schema is self-contained. One shape file covers the complete pattern:

| Shape file | Validates |
|---|---|
| `specs/shape.ttl` | Process realizes a TensileTestMethod and has a specimen input; each datum has exactly one value specification; each SVS has a positive numeric value, a unit IRI, and quality links |

In [7]:
conforms, violations = schema.validate(flat)

print(f"Conforms: {conforms}")
for v in violations:
    print(f"  Violation: {v}")

Conforms: False
  Violation: A ScalarValueSpecification must have a positive numeric value via pmdco:PMD_0000006.  [PMD_0000006]
  Violation: A ScalarValueSpecification must have a positive numeric value via pmdco:PMD_0000006.  [PMD_0000006]
  Violation: A ScalarValueSpecification must have a positive numeric value via pmdco:PMD_0000006.  [PMD_0000006]
  Violation: A ScalarValueSpecification must have a positive numeric value via pmdco:PMD_0000006.  [PMD_0000006]
  Violation: A ScalarValueSpecification must have a positive numeric value via pmdco:PMD_0000006.  [PMD_0000006]


## Step 5: Inspect the results

The SPARQL query below extracts the mechanical properties captured in the
graph and prints them as a table.

You do not need to understand SPARQL to use this notebook. Just run the cell
and check that the values match what you entered.

In [8]:
PMDCO = rdflib.Namespace("https://w3id.org/pmd/co/")
OBO   = rdflib.Namespace("http://purl.obolibrary.org/obo/")

proc_iri = next(flat.subjects(rdflib.RDF.type, PMDCO["PMD_0000974"]))
print(f"Test IRI : {proc_iri}")
print(f"Label    : {flat.value(proc_iri, rdflib.RDFS.label)}")
print()

# Each result is a ScalarMeasurementDatum (IAO_0000109) linked via OBI_0000299.
# The datum carries its SVS (OBI_0001938) which holds value, optional unit, and quality link.
# Strain rate (TTO_0000051) has no unit IRI, following the TTO reference data pattern.
SPARQL = """
PREFIX obi:   <http://purl.obolibrary.org/obo/OBI_>
PREFIX iao:   <http://purl.obolibrary.org/obo/IAO_>
PREFIX obo:   <http://purl.obolibrary.org/obo/>
PREFIX pmdco: <https://w3id.org/pmd/co/>

SELECT ?propertyType ?value ?unit
WHERE {
  ?test  a              pmdco:PMD_0000974 ;
         obi:0000299    ?smd .
  ?smd   a              obo:IAO_0000109 ;
         obi:0001938    ?svs .
  ?svs   a              obi:0001931 ;
         pmdco:PMD_0000006  ?value ;
         obi:0001927    ?prop .
  ?prop  a              ?propertyType .
  OPTIONAL { ?svs iao:0000039 ?unit . }
}
ORDER BY ?propertyType
"""

rows = list(flat.query(SPARQL))
if rows:
    print(f"{'Property class':<45}  {'Value':>10}  Unit")
    print("-" * 80)
    for r in rows:
        prop = str(r.propertyType).rsplit("/", 1)[-1].rsplit("#", 1)[-1]
        unit = str(r.unit).rsplit("/", 1)[-1] if r.unit else "—"
        print(f"{prop:<45}  {float(r.value):>10.6g}  {unit}")
else:
    print("No measured properties found in the graph.")

Test IRI : https://example.org/tensile-test-tensile-test-316l-bar-1
Label    : Tensile test 316L bar 1



Property class                                      Value  Unit
--------------------------------------------------------------------------------
PMD_0000967                                            23  DEG_C
TTO_0000009                                           310  MegaPa
TTO_0000033                                            40  PERCENT
TTO_0000038                                            65  PERCENT
TTO_0000051                                       0.00025  —
TTO_0000053                                           620  MegaPa


## Summary

| Step | What happens |
|---|---|
| 1 | You fill in `example.input.json` with the test name, specimen IRI, conditions, and results |
| 2 | The data is converted to an OO-LD document (ontology-mapped JSON) following the TTO pattern |
| 3 | The OO-LD is parsed into an RDF graph (74 triples for the example with 4 results and 2 conditions) |
| 4 | The graph is validated against one self-contained SHACL shape file |
| 5 | Measured properties and conditions are extracted from the graph to confirm correctness |

To test a different specimen, edit `docs/example.input.json` and re-run all cells.

---

## What comes next

This notebook used a hand-crafted JSON file as input. In practice, your measurement
data lives in an instrument export file, not a JSON you wrote by hand.

[Notebook 2: from machine file to RDF](2_tensile_test_csv_workflow.ipynb) shows how
to replace the manual input with a real Zwick/Roell export. The schema, the transform,
the SHACL shapes, and the RDF structure are identical; only the data source changes.

---

## Further reading

- [TTO repository](https://github.com/materialdigital/tensile-test-ontology): reference data and ontology
- [PMDCo measurement pattern](https://github.com/materialdigital/core-ontology/tree/main/patterns/measurement)
- [Tensile Test (PMDCo)](../../PMDCo/README.md): the label-based variant of this schema
- [OO-LD primer](../../../../docs/2_oold-primer.md): what OO-LD is and how it works
- [Schema format reference](../../../../docs/3_schema-format.md): folder structure and naming conventions
